# 📈 Sesiunea 3 - Linear Regression

La cursul de Python ați explorat și curățat Titanic: l-ați descris, grupat, ați tratat valori lipsă. Până acum am **descris** datele.

Azi punem prima întrebare de **predicție**: **cât** a costat biletul unui pasager? Răspunsul e un *număr*, iar instrumentul nou se numește **regresie liniară**.

**Cum lucrăm:** întâi recapitulăm datele aici, mergem la slide-uri pentru matematică, apoi revenim să antrenăm modelul. Ca de obicei: tu ghicești, apoi verificăm în cod.

## Partea 1 · Recap pe Titanic (înainte de slide-uri) 🔍

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", 100)
plt.rcParams["figure.figsize"] = (8, 4.5)

# același Titanic pe care l-ați folosit la cursul de Python
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)
print("shape:", df.shape)
df.head()

## 1. Ce vrem să prezicem azi?

Avem 891 de pasageri și 12 coloane. Azi vrem să prezicem un **număr**. Care coloană crezi că e ținta?

<details>
<summary><b>💡 Răspuns (deschide după ce ai ghicit)</b></summary>
<br>
Ținta e **`Fare`** (prețul biletului), o valoare continuă, exact ce are nevoie regresia liniară. O coloană ca `Survived` (0/1) ar fi o *categorie*, adică o problemă de clasificare, altă poveste.
</details>

In [ ]:
# ținta: Fare. Feature-urile candidate: ce știm despre pasager.
df[["Pclass", "Age", "SibSp", "Parch", "Sex", "Fare"]].head()

## 2. Cât de variat e prețul?

💭 Ghicești: cât a fost cel mai scump bilet? Și de ce media e mult mai mare decât mediana?

<details>
<summary><b>💡 Răspuns (deschide după ce ai ghicit)</b></summary>
<br>
Cel mai scump: **£512** (o avere în 1912). Media (~£32) e mult peste mediană (~£14) pentru că **câteva bilete uriașe trag media în sus**. E o distribuție înclinată, exact ca prețurile din viața reală.
</details>

In [ ]:
print(df["Fare"].describe().round(2))
print("\nmediană:", df["Fare"].median())

## 3. Ce influențează prețul?

🗳️ Ghicești: care variabilă e cel mai puternic legată de preț? (A) vârsta (B) clasa (C) numărul de rude

<details>
<summary><b>💡 Răspuns (deschide după ce ai ghicit)</b></summary>
<br>
**Clasa (`Pclass`)**, de departe. Corelația e negativă (~ −0.55): clasă mai „mare" ca număr (clasa 3) înseamnă preț mai mic. Vârsta aproape că nu contează.
</details>

In [ ]:
num = df[["Fare", "Pclass", "Age", "SibSp", "Parch"]]
print(num.corr()["Fare"].round(3).sort_values())

## 4. Să *vedem* relația

De ce fac un grafic dacă am deja corelația? 📈

<details>
<summary><b>💡 Răspuns (deschide după ce ai ghicit)</b></summary>
<br>
Cifra spune *că* există o relație; graficul arată *ce formă* are. Aici vezi clar o **tendință descrescătoare**: clasa 1 = bilete scumpe, clasa 3 = ieftine. Exact relația pe care regresia o va rezuma printr-o dreaptă.
</details>

In [ ]:
# jitter pe axa x doar ca punctele să nu se suprapună
jitter = df["Pclass"] + np.random.normal(0, 0.06, len(df))
plt.scatter(jitter, df["Fare"], s=16, alpha=0.35)
plt.xticks([1, 2, 3]); plt.xlabel("Clasa (Pclass)"); plt.ylabel("Preț bilet (Fare)")
plt.title("Prețul biletului scade cu clasa")
plt.ylim(-10, 300); plt.show()

### 🎯 Pauză: trecem la slide-uri

Știm *ce* vrem: o dreaptă care trece prin norul ăsta de puncte. Mergem la prezentare să vedem **cum** o găsește modelul (ecuația, reziduuri, cele mai mici pătrate), apoi ne întoarcem aici.

## Partea 2 · Regresia liniară în cod (după slide-uri) 🏗️

## 5. Pregătim datele și le împărțim

Vrem `Fare` din `Pclass`. Ce trebuie curățat înainte? Și de ce împărțim datele chiar acum? 🤔

<details>
<summary><b>💡 Răspuns (deschide după ce ai ghicit)</b></summary>
<br>
Curățare: aproape nimic, `Pclass` și `Fare` sunt complete și numerice. Împărțim în **train** (antrenare) și **test** (evaluare) *de la început*, ca să antrenăm doar pe train și să ținem test-ul curat pentru o notă cinstită la final. *Notă: dacă am adăuga `Age` sau `Sex`, ar trebui întâi să tratăm goluri și text, dar aia e pentru data viitoare.*
</details>

In [ ]:
X = df[["Pclass"]]   # feature-ul (ce folosim ca să prezicem)
y = df["Fare"]        # ținta (ce prezicem)
print("goluri în X:", int(X.isnull().sum().sum()), " | goluri în y:", int(y.isnull().sum()))

# împărțim: 80% antrenare, 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
print("train:", X_train.shape[0], "pasageri  |  test:", X_test.shape[0], "pasageri")

## 6. Antrenăm dreapta (regresie simplă)

Antrenăm `Fare = a + b · Pclass` **pe train**. 💭 Ghicești semnul pantei `b`: pozitiv sau negativ?

<details>
<summary><b>💡 Răspuns (deschide după ce ai ghicit)</b></summary>
<br>
**Negativ.** Când numărul clasei crește (1 → 3), prețul scade. Modelul găsește o pantă de ~**−32**: fiecare clasă în jos taie ~£32 din preț. Interceptul (~106) e doar punctul teoretic de pornire al dreptei.
</details>

In [ ]:
model = LinearRegression().fit(X_train, y_train)   # învață doar din train
a = model.intercept_
b = model.coef_[0]
print(f"Fare = {a:.1f}  +  ({b:.1f}) * Pclass")
print(f"\nInterpretare: fiecare clasă în jos schimbă prețul cu {b:.1f} lire.")

## 7. Desenăm dreapta învățată

Punem dreapta găsită de model peste norul de puncte. Trece ea „prin mijloc"?

In [ ]:
jitter = df["Pclass"] + np.random.normal(0, 0.06, len(df))
plt.scatter(jitter, y, s=16, alpha=0.30, label="pasageri reali")

xs = np.linspace(0.7, 3.3, 50)
line = model.predict(pd.DataFrame({"Pclass": xs}))   # DataFrame cu același nume de coloană
plt.plot(xs, line, color="black", lw=3, label="dreapta învățată")

plt.xticks([1, 2, 3]); plt.xlabel("Clasa (Pclass)"); plt.ylabel("Preț bilet (Fare)")
plt.ylim(-10, 300); plt.legend(); plt.show()

## 8. Cât de bine prezice? (pe test)

Am antrenat pe train. Acum îl notăm pe **test**, date pe care nu le-a văzut. De ce nu-l notăm pe train? 🤔

<details>
<summary><b>💡 Răspuns (deschide după ce ai ghicit)</b></summary>
<br>
Pentru că pe train modelul le-a „văzut" deja, ar fi ca și cum te-ai nota singur la propriul examen. Nota reală vine din date noi. Cu o singură variabilă, ne așteptăm la un R² modest (~0.37).
</details>

In [ ]:
pred = model.predict(X_test)   # același model antrenat pe train, evaluat pe test

mae  = mean_absolute_error(y_test, pred)
rmse = mean_squared_error(y_test, pred) ** 0.5
r2   = r2_score(y_test, pred)
print(f"MAE  = {mae:.1f} lire   (in medie, cam pe atat gresim)")
print(f"RMSE = {rmse:.1f} lire   (penalizeaza greselile mari)")
print(f"R2   = {r2:.3f}   (~{r2*100:.0f}% din variatia pretului explicata)")

## 9. Predicție vs realitate

Facem un grafic: prețul real (pe x) vs prețul prezis (pe y). 💭 De ce crezi că apar doar **trei** valori posibile de predicție?

<details>
<summary><b>💡 Răspuns (deschide după ce ai ghicit)</b></summary>
<br>
Pentru că avem o **singură** variabilă, `Pclass`, cu doar 3 valori (1, 2, 3). Deci modelul poate da doar 3 predicții, câte una per clasă. De aici limita regresiei simple, și motivul pentru care **data viitoare adăugăm mai multe variabile** (regresie multiplă), ca să prindem și diferențele din interiorul aceleiași clase.
</details>

In [ ]:
plt.scatter(y_test, pred, s=18, alpha=0.5)
lim = [0, 300]
plt.plot(lim, lim, color="black", lw=2, ls="--", label="predicție perfectă")
plt.xlabel("Preț real"); plt.ylabel("Preț prezis")
plt.xlim(lim); plt.ylim(lim); plt.legend(); plt.title("Predicție vs realitate")
plt.show()

## 10. Ce am învățat 🎓

Am făcut primul nostru model: am prezis un **număr** (prețul biletului) cu o **dreaptă**.

- `Fare = a + b · Pclass`, și am înțeles ce înseamnă panta și interceptul,
- am antrenat cu `.fit()` și am desenat dreapta învățată,
- am evaluat cinstit pe date de **test**: MAE, RMSE, R² (~0.37 cu o singură variabilă),
- lecția de onestitate: cu un singur feature, modelul dă doar 3 predicții.

**Data viitoare:** adăugăm mai multe variabile deodată, **regresie multiplă**, și vedem cum crește R². 🚀